# 07 — Kaggle Submission — Batched Inference

This notebook generates Monet-style paintings from competition photos using the original authors’ pretrained `style_monet` generator, without training or fine-tuning.


## Want to see more?

I turned this Kaggle competition into a larger project. You can follow my work in the [GitHub repository](https://github.com/KurtMeehanPro/painter-myself).

Now back to this notebook...


## Competition output

The [competition format](https://www.kaggle.com/competitions/gan-getting-started/overview) calls for 7,000–10,000 RGB JPGs at 256×256. The generated paintings are collected in one output ZIP, `images.zip`.


## Pretrained model

The authors’ [model zoo](https://github.com/junyanz/CycleGAN#model-zoo) identifies `style_monet` as landscape photo → Monet style. The checkpoint comes from their [official downloader](https://github.com/junyanz/pytorch-CycleGAN-and-pix2pix/blob/master/scripts/download_cyclegan_model.sh). Each output is generated from a competition photo.


## Imports


In [4]:
# standard library
from io import BytesIO
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile, ZIP_STORED
import functools
import time

# third-party
import numpy as np
from PIL import Image
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader


## Configuration

One environment switch selects local project paths or Kaggle inputs and output paths. Local loading uses the notebook process; Kaggle loading uses two workers.


In [ ]:
RUN_ENVIRONMENT = "kaggle"  # Change to "kaggle" before uploading to Kaggle.
ALLOW_CHECKPOINT_DOWNLOAD = True
BATCH_SIZE = 16
PHOTO_DIR_OVERRIDE = None
CHECKPOINT_PATH_OVERRIDE = None

if RUN_ENVIRONMENT == "local":
    project_root = next(
        (path for path in (Path.cwd(), *Path.cwd().parents)
         if (path / "scripts" / "painter_dataset_config.py").is_file()),
        None,
    )
    if project_root is None:
        raise FileNotFoundError("Project root not found from the current notebook directory.")
    input_root = project_root / "data" / "raw"
    output_dir = project_root / "data" / "outputs" / "style_monet_batched"
    scratch_dir = output_dir / "tmp"
    NUM_WORKERS = 0  # In-process loading avoids macOS notebook multiprocessing issues.
elif RUN_ENVIRONMENT == "kaggle":
    input_root = Path("/kaggle/input")
    output_dir = Path("/kaggle/working")
    scratch_dir = Path("/kaggle/tmp/style_monet")
    NUM_WORKERS = 2
else:
    raise ValueError(f"Unsupported RUN_ENVIRONMENT: {RUN_ENVIRONMENT!r}; expected 'local' or 'kaggle'.")

checkpoint_url = "https://efrosgans.eecs.berkeley.edu/cyclegan/pretrained_models/style_monet.pth"
output_zip_path = output_dir / "images.zip"
temporary_output_zip_path = scratch_dir / "images.partial.zip"


## Competition photos
- find the competition photos.
- checks the selected environment’s input folder
- either uses a folder specified or searches for the photo folder automatically
- checks how many photos it found

In [6]:
if not input_root.is_dir():
    raise FileNotFoundError(f"Photo input directory not found: {input_root}")
if PHOTO_DIR_OVERRIDE is not None:
    photo_dir = Path(PHOTO_DIR_OVERRIDE)
elif RUN_ENVIRONMENT == "local":
    photo_dir = input_root / "photo_jpg"
else:
    photo_dirs = sorted(path for path in input_root.rglob("photo_jpg") if path.is_dir())
    if len(photo_dirs) != 1:
        raise ValueError(f"Expected one photo_jpg directory; found {photo_dirs}. Attach the competition input or set PHOTO_DIR_OVERRIDE.")
    photo_dir = photo_dirs[0]
photo_paths = sorted(photo_dir.glob("*.jpg"))
if not 7000 <= len(photo_paths) <= 10000:
    raise ValueError(f"Expected 7,000–10,000 photos; found {len(photo_paths):,} in {photo_dir}.")
print(f"Photos: {len(photo_paths):,} — {photo_dir}")


Photos: 7,038 — /Users/KurtMeehanPro/Documents/code/git/painter-myself/data/raw/photo_jpg


## Pretrained weights

Local runs reuse the project checkpoint. Kaggle runs use attached weights when available. Missing weights can download from the authors’ site.


In [7]:
scratch_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)
if CHECKPOINT_PATH_OVERRIDE is not None:
    checkpoint_path = Path(CHECKPOINT_PATH_OVERRIDE)
elif RUN_ENVIRONMENT == "local":
    checkpoint_path = project_root / "data" / "checkpoints" / "style_monet" / "style_monet.pth"
else:
    candidates = sorted(input_root.rglob("style_monet.pth"))
    if len(candidates) > 1:
        raise ValueError(f"Multiple checkpoints found: {candidates}. Set CHECKPOINT_PATH_OVERRIDE.")
    checkpoint_path = candidates[0] if candidates else scratch_dir / "style_monet.pth"

if not checkpoint_path.is_file():
    if not ALLOW_CHECKPOINT_DOWNLOAD:
        raise FileNotFoundError("Attach style_monet.pth as an input, or enable Kaggle Internet and ALLOW_CHECKPOINT_DOWNLOAD.")
    if CHECKPOINT_PATH_OVERRIDE is not None:
        raise FileNotFoundError(f"Checkpoint override not found: {checkpoint_path}")
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    download_path = scratch_dir / "style_monet.download"
    try:
        urlretrieve(checkpoint_url, download_path)
    except Exception as e:
        raise RuntimeError("Checkpoint download failed. Check Kaggle Internet or attach style_monet.pth for offline use.") from e
    download_path.replace(checkpoint_path)

print(f"Checkpoint: {checkpoint_path}")


Checkpoint: /Users/KurtMeehanPro/Documents/code/git/painter-myself/data/checkpoints/style_monet/style_monet.pth


## Generator implementation

The original authors’ generator uses nine residual blocks, 64 filters in the first convolution, and InstanceNorm. The two class definitions are reused unchanged; the original license appears below.


In [8]:
class ResnetGenerator(nn.Module):
    """Resnet-based generator that consists of Resnet blocks between a few downsampling/upsampling operations.

    We adapt Torch code and idea from Justin Johnson's neural style transfer project(https://github.com/jcjohnson/fast-neural-style)
    """

    def __init__(self, input_nc, output_nc, ngf=64, norm_layer=nn.BatchNorm2d, use_dropout=False, n_blocks=6, padding_type="reflect"):
        """Construct a Resnet-based generator

        Parameters:
            input_nc (int)      -- the number of channels in input images
            output_nc (int)     -- the number of channels in output images
            ngf (int)           -- the number of filters in the last conv layer
            norm_layer          -- normalization layer
            use_dropout (bool)  -- if use dropout layers
            n_blocks (int)      -- the number of ResNet blocks
            padding_type (str)  -- the name of padding layer in conv layers: reflect | replicate | zero
        """
        assert n_blocks >= 0
        super(ResnetGenerator, self).__init__()
        if type(norm_layer) == functools.partial:
            use_bias = norm_layer.func == nn.InstanceNorm2d
        else:
            use_bias = norm_layer == nn.InstanceNorm2d

        model = [nn.ReflectionPad2d(3), nn.Conv2d(input_nc, ngf, kernel_size=7, padding=0, bias=use_bias), norm_layer(ngf), nn.ReLU(True)]

        n_downsampling = 2
        for i in range(n_downsampling):  # add downsampling layers
            mult = 2**i
            model += [nn.Conv2d(ngf * mult, ngf * mult * 2, kernel_size=3, stride=2, padding=1, bias=use_bias), norm_layer(ngf * mult * 2), nn.ReLU(True)]

        mult = 2**n_downsampling
        for i in range(n_blocks):  # add ResNet blocks

            model += [ResnetBlock(ngf * mult, padding_type=padding_type, norm_layer=norm_layer, use_dropout=use_dropout, use_bias=use_bias)]

        for i in range(n_downsampling):  # add upsampling layers
            mult = 2 ** (n_downsampling - i)
            model += [nn.ConvTranspose2d(ngf * mult, int(ngf * mult / 2), kernel_size=3, stride=2, padding=1, output_padding=1, bias=use_bias), norm_layer(int(ngf * mult / 2)), nn.ReLU(True)]
        model += [nn.ReflectionPad2d(3)]
        model += [nn.Conv2d(ngf, output_nc, kernel_size=7, padding=0)]
        model += [nn.Tanh()]

        self.model = nn.Sequential(*model)

    def forward(self, input):
        """Standard forward"""
        return self.model(input)


In [9]:
class ResnetBlock(nn.Module):
    """Define a Resnet block"""

    def __init__(self, dim, padding_type, norm_layer, use_dropout, use_bias):
        """Initialize the Resnet block

        A resnet block is a conv block with skip connections
        We construct a conv block with build_conv_block function,
        and implement skip connections in <forward> function.
        Original Resnet paper: https://arxiv.org/pdf/1512.03385.pdf
        """
        super(ResnetBlock, self).__init__()
        self.conv_block = self.build_conv_block(dim, padding_type, norm_layer, use_dropout, use_bias)

    def build_conv_block(self, dim, padding_type, norm_layer, use_dropout, use_bias):
        """Construct a convolutional block.

        Parameters:
            dim (int)           -- the number of channels in the conv layer.
            padding_type (str)  -- the name of padding layer: reflect | replicate | zero
            norm_layer          -- normalization layer
            use_dropout (bool)  -- if use dropout layers.
            use_bias (bool)     -- if the conv layer uses bias or not

        Returns a conv block (with a conv layer, a normalization layer, and a non-linearity layer (ReLU))
        """
        conv_block = []
        p = 0
        if padding_type == "reflect":
            conv_block += [nn.ReflectionPad2d(1)]
        elif padding_type == "replicate":
            conv_block += [nn.ReplicationPad2d(1)]
        elif padding_type == "zero":
            p = 1
        else:
            raise NotImplementedError("padding [%s] is not implemented" % padding_type)

        conv_block += [nn.Conv2d(dim, dim, kernel_size=3, padding=p, bias=use_bias), norm_layer(dim), nn.ReLU(True)]
        if use_dropout:
            conv_block += [nn.Dropout(0.5)]

        p = 0
        if padding_type == "reflect":
            conv_block += [nn.ReflectionPad2d(1)]
        elif padding_type == "replicate":
            conv_block += [nn.ReplicationPad2d(1)]
        elif padding_type == "zero":
            p = 1
        else:
            raise NotImplementedError("padding [%s] is not implemented" % padding_type)
        conv_block += [nn.Conv2d(dim, dim, kernel_size=3, padding=p, bias=use_bias), norm_layer(dim)]

        return nn.Sequential(*conv_block)

    def forward(self, x):
        """Forward function (with skip connections)"""
        out = x + self.conv_block(x)  # add skip connections
        return out


## Load the generator

The loader omits obsolete InstanceNorm bookkeeping buffers and strictly loads every learned parameter. Parameters are frozen and the generator runs in evaluation mode.


In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else
                      "mps" if torch.backends.mps.is_available() else "cpu")
generator = ResnetGenerator(3, 3, ngf=64, norm_layer=nn.InstanceNorm2d, n_blocks=9)
state = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
expected = generator.state_dict()
# Older checkpoints contain InstanceNorm statistics unused by this architecture.
obsolete = [key for key in state if key not in expected]
for key in obsolete:
    module_name, buffer_name = key.rsplit(".", 1)
    module = generator.get_submodule(module_name)
    if not isinstance(module, nn.InstanceNorm2d) or buffer_name not in (
        "running_mean", "running_var", "num_batches_tracked"
    ):
        raise ValueError(f"Unexpected checkpoint entry: {key}")
    del state[key]
generator.load_state_dict(state, strict=True)
generator.requires_grad_(False)
generator.eval()
generator.to(device)
if device.type == "cuda" and torch.cuda.device_count() > 1:
    generator = nn.DataParallel(generator)
    generator.eval()
print(f"Loaded {len(state)} parameter tensors; omitted {len(obsolete)} legacy buffers.")
print(f"Device: {device}; PyTorch: {torch.__version__}")


Loaded 48 parameter tensors; omitted 69 legacy buffers.
Device: mps; PyTorch: 2.14.0


## Photo-to-painting conversion

RGB pixels are normalized to [-1, 1] for inference. The generated pixels are converted back to uint8 RGB images, without random cropping or flipping.


In [11]:
class PhotoDataset(Dataset):
    """Load RGB photos as normalized 256×256 tensors with source filenames."""

    def __init__(self, paths):
        self.paths = paths

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        image_path = self.paths[index]
        with Image.open(image_path) as image:
            image = image.convert("RGB")
            if image.size != (256, 256):
                image = image.resize((256, 256), Image.Resampling.BICUBIC)
            pixels = np.asarray(image, dtype=np.float32) / 127.5 - 1.0
        tensor = torch.from_numpy(pixels.transpose(2, 0, 1).copy())
        return tensor, image_path.name


photo_loader = DataLoader(
    PhotoDataset(photo_paths),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=device.type == "cuda",
)


## Generate the output ZIP

Each photo is transformed once. Generated JPG bytes are written directly into the output ZIP, with temporary files kept in the selected scratch folder.


In [12]:
start = time.perf_counter()
processed = 0
next_report = 500
with ZipFile(temporary_output_zip_path, "w", compression=ZIP_STORED) as output_zip:
    for tensors, filenames in photo_loader:
        tensors = tensors.to(device, non_blocking=device.type == "cuda")
        with torch.inference_mode():
            results = generator(tensors).cpu()
        if not torch.isfinite(results).all():
            raise ValueError(f"Nonfinite output for batch: {filenames}")
        pixels = ((results.permute(0, 2, 3, 1).numpy() + 1.0) * 127.5).clip(0, 255).astype(np.uint8)
        for image_pixels, filename in zip(pixels, filenames):
            painting = Image.fromarray(image_pixels)
            if painting.size != (256, 256) or painting.mode != "RGB":
                raise ValueError(f"Invalid generated image: {filename}")
            buffer = BytesIO()
            painting.save(buffer, format="JPEG", quality=95)
            output_zip.writestr(filename, buffer.getvalue())
        processed += len(filenames)
        if processed >= next_report or processed == len(photo_paths):
            print(f"Generated {processed:,}/{len(photo_paths):,} in {time.perf_counter() - start:.1f}s", flush=True)
            next_report += 500


Generated 512/7,038 in 5.2s
Generated 1,008/7,038 in 9.8s
Generated 1,504/7,038 in 14.4s
Generated 2,000/7,038 in 19.1s
Generated 2,512/7,038 in 23.7s
Generated 3,008/7,038 in 28.2s
Generated 3,504/7,038 in 32.7s
Generated 4,000/7,038 in 37.2s
Generated 4,512/7,038 in 41.9s
Generated 5,008/7,038 in 46.4s
Generated 5,504/7,038 in 51.0s
Generated 6,000/7,038 in 55.5s
Generated 6,512/7,038 in 60.1s
Generated 7,008/7,038 in 64.6s
Generated 7,038/7,038 in 64.9s


## Validate the output ZIP

Validation checks filenames, file count, ZIP integrity, and every generated image’s RGB mode and 256×256 dimensions. The completed output is then published as `images.zip` in the selected output folder.


In [13]:
with ZipFile(temporary_output_zip_path) as output_zip:
    names = output_zip.namelist()
    assert len(names) == len(set(names)) == len(photo_paths)
    assert set(names) == {path.name for path in photo_paths}
    assert output_zip.testzip() is None
    for name in names:
        with Image.open(BytesIO(output_zip.read(name))) as image:
            assert image.format == "JPEG" and image.mode == "RGB" and image.size == (256, 256), name
            image.load()
print(f"Validated {len(names):,} RGB JPEGs, all 256×256.")


Validated 7,038 RGB JPEGs, all 256×256.


In [14]:
# Copy across filesystems if scratch and output storage differ.
from shutil import copyfile

copyfile(temporary_output_zip_path, output_zip_path)
print(f"Output ZIP: {output_zip_path} ({output_zip_path.stat().st_size / 1024**2:.1f} MiB)")


Output ZIP: /Users/KurtMeehanPro/Documents/code/git/painter-myself/data/outputs/style_monet_batched/images.zip (223.0 MiB)


## Attribution and license

This notebook reuses the generator implementation from the original CycleGAN authors. Their generator code is used alongside my own code in this notebook.

Attribution for the Generator Code: Jun-Yan Zhu and Taesung Park, [pytorch-CycleGAN-and-pix2pix](https://github.com/junyanz/pytorch-CycleGAN-and-pix2pix).

<details>
<summary>Original license</summary>

```text
Copyright (c) 2017, Jun-Yan Zhu and Taesung Park
All rights reserved.

Redistribution and use in source and binary forms, with or without
modification, are permitted provided that the following conditions are met:

* Redistributions of source code must retain the above copyright notice, this
  list of conditions and the following disclaimer.

* Redistributions in binary form must reproduce the above copyright notice,
  this list of conditions and the following disclaimer in the documentation
  and/or other materials provided with the distribution.

THIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS "AS IS"
AND ANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE
IMPLIED WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE ARE
DISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT HOLDER OR CONTRIBUTORS BE LIABLE
FOR ANY DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR CONSEQUENTIAL
DAMAGES (INCLUDING, BUT NOT LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR
SERVICES; LOSS OF USE, DATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER
CAUSED AND ON ANY THEORY OF LIABILITY, WHETHER IN CONTRACT, STRICT LIABILITY,
OR TORT (INCLUDING NEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY OUT OF THE USE
OF THIS SOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGE.


--------------------------- LICENSE FOR pix2pix --------------------------------
BSD License

For pix2pix software
Copyright (c) 2016, Phillip Isola and Jun-Yan Zhu
All rights reserved.

Redistribution and use in source and binary forms, with or without
modification, are permitted provided that the following conditions are met:

* Redistributions of source code must retain the above copyright notice, this
  list of conditions and the following disclaimer.

* Redistributions in binary form must reproduce the above copyright notice,
  this list of conditions and the following disclaimer in the documentation
  and/or other materials provided with the distribution.

----------------------------- LICENSE FOR DCGAN --------------------------------
BSD License

For dcgan.torch software

Copyright (c) 2015, Facebook, Inc. All rights reserved.

Redistribution and use in source and binary forms, with or without modification, are permitted provided that the following conditions are met:

Redistributions of source code must retain the above copyright notice, this list of conditions and the following disclaimer.

Redistributions in binary form must reproduce the above copyright notice, this list of conditions and the following disclaimer in the documentation and/or other materials provided with the distribution.

Neither the name Facebook nor the names of its contributors may be used to endorse or promote products derived from this software without specific prior written permission.

THIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS "AS IS" AND ANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE IMPLIED WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE ARE DISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT HOLDER OR CONTRIBUTORS BE LIABLE FOR ANY DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR CONSEQUENTIAL DAMAGES (INCLUDING, BUT NOT LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR SERVICES; LOSS OF USE, DATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER CAUSED AND ON ANY THEORY OF LIABILITY, WHETHER IN CONTRACT, STRICT LIABILITY, OR TORT (INCLUDING NEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY OUT OF THE USE OF THIS SOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGE.

```

</details>
